In [0]:
"""
03_quality_events.py

Creates the Silver Quality Events table.

Input:
    parsed_events

Output:
    quality_events

Author:
Sumanth Vempalle

Version:
2.1.0
"""

from pyspark import pipelines as dp

from pyspark.sql.functions import col


# ============================================================
# Quality Events
# ============================================================

@dp.table(
    name="quality_events",
    comment="Validated manufacturing quality events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dp.expect_or_drop(
    "valid_execution_id",
    "execution_id IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_serial_number",
    "serial_number IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_test_program",
    "test_program_id IS NOT NULL",
)

@dp.expect(
    "valid_result",
    "result IN ('PASS', 'FAIL')",
)

def quality_events():

    df = dp.read_stream("parsed_events")

    return (

        df

        # -----------------------------------------
        # Keep only quality events
        # -----------------------------------------

        .filter(
            col("event_type") == "QUALITY_COMPLETED"
        )

        # -----------------------------------------
        # Business columns
        # -----------------------------------------

        .select(

            "event_id",
            "event_timestamp",
            "event_version",

            "plant_code",

            "execution_id",
            "serial_number",

            "product_code",

            "source_system",
            "correlation_id",

            "bronze_ingestion_timestamp",
            "silver_processing_timestamp",

            "payload.test_result_id",
            "payload.test_program_id",
            "payload.test_name",

            "payload.target_value",
            "payload.measured_value",

            "payload.unit",

            "payload.result",

            "payload.product_name",
            "payload.family",

        )

    )